# 重回帰分析

単回帰が「1 つの入力で 1 つの出力を説明する」モデルだとすると、重回帰は「複数の事情が同時に効いている現実」に少し近づいたモデルです。面積だけで家賃を決めるのではなく、立地や築年数も一緒に見る、という発想に近いです。

このノートでは、複数特徴量を同時に使うと係数の意味がどう変わるか、行列の式は何をしているのか、そして入力どうしが似すぎていると何が起こるのかを見ます。


## まずは「理由が 1 つではない」状況を考える

現実の予測問題では、1 つの原因だけで結果が決まることはほとんどありません。重回帰の出発点は、複数の事情が同時に効いている状況で、それぞれの寄与をどう切り分けるかという問いです。

そのぶん、単回帰より少しだけ読むべきものが増えます。係数の符号や大きさだけでなく、変数どうしが似すぎていないかにも注意が必要です。


このノートでは、まず複数の事情が混ざった疑似データを作り、次にそれを 1 本の式へまとめます。そのうえで係数を求め、『この係数はどこまで素直に読んでよいのか』を VIF で確かめます。

流れとしては、表現力を増やした代わりに、解釈の注意点も増えることを体験する構成です。


## このノートで使う基本語彙

- 重回帰: 複数の入力特徴量を同時に使う回帰
- 切片: すべての説明変数が 0 のときの基準値
- 係数: 他の変数を固定したとき、その特徴量が 1 増えると予測がどれだけ変わるか
- 多重共線性: 入力どうしが似すぎていて、係数の解釈が不安定になる状態

予測だけならそこまで困らなくても、係数を「意味のある寄与」として読みたいときに多重共線性は強く効きます。


## 係数を出して終わりにしない

ここでは、係数がデータ生成式に近いかを見るだけでなく、その係数がどれくらい安心して読めるかも確認します。重回帰では「計算できた」と「解釈してよい」が必ずしも同じではありません。


## 出力を見るときの視点

`beta_hat` は、各特徴量の寄与を表す推定値です。正なら増やす方向、負なら減らす方向に効いている、とまず読めば十分です。

ただし、値が安定しているかどうかは VIF も合わせて見ないと判断しづらい、というのがこのノートの後半です。


## 行列の式に飲まれないために

`(X^T X)^(-1) X^T y` の形は intimidating ですが、やっていることは「全データをまとめて見て、いちばん外れが小さくなる係数を求める」ことです。数式の見た目より、`X` が何列あり、その列同士がどれくらい似ているかの方が実務では重要です。


## どこまでをこのノートで扱うか

ここでは、最小二乗解と多重共線性の入口に集中します。変数選択の戦略や正則化は、別のノートで本格的に扱う前提です。


## まずは、複数の要因が効くデータを作る

最初に `x1` と `x2` が両方とも `y` に効く疑似データを作ります。ここでは「2 つの入力を一緒に見ないと、片方の意味を読み違えやすい」状況をわざと作っています。


In [ ]:
import numpy as np
np.random.seed(11)

n = 120
x1 = np.random.randn(n)
x2 = 0.6 * x1 + 0.8 * np.random.randn(n)  # 相関あり
eps = 0.5 * np.random.randn(n)
y = 1.2 + 2.0 * x1 - 1.5 * x2 + eps

X = np.column_stack([np.ones(n), x1, x2])


## まとめて解くと、係数はどう出るか

次は行列表現を使って、係数を一括で求めます。先頭の 1 の列は切片のための列で、直線を原点固定にしないために入っています。


In [ ]:
# 最小二乗解: beta = (X^T X)^(-1) X^T y
beta = np.linalg.pinv(X.T @ X) @ X.T @ y
pred = X @ beta
mse = np.mean((pred - y) ** 2)

print('beta_hat =', np.round(beta, 6))
print('train MSE=', round(mse, 6))


## 入力どうしが似すぎていないかも見る

係数が出たあとで VIF を見るのは、予測精度ではなく「その係数をどこまで信用して読めるか」を確認するためです。似た説明変数が並ぶと、どちらが効いたのかを切り分けにくくなります。


In [ ]:
# VIFで多重共線性を確認
# VIF_j = 1 / (1 - R_j^2)

def vif(x_target, x_other):
    Xo = np.column_stack([np.ones(len(x_other)), x_other])
    b = np.linalg.pinv(Xo.T @ Xo) @ Xo.T @ x_target
    pred = Xo @ b
    ss_res = np.sum((x_target - pred) ** 2)
    ss_tot = np.sum((x_target - x_target.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    return 1.0 / (1.0 - r2)

vif_x1 = vif(x1, x2)
vif_x2 = vif(x2, x1)
print('VIF(x1)=', round(vif_x1, 6), 'VIF(x2)=', round(vif_x2, 6))


重回帰では、係数推定と同じくらい「その係数が安定しているか」を見ることが大切です。VIF を併せて見るのは、予測モデルとしてだけでなく説明モデルとして使うときの最低限の作法です。
